In [ ]:
import os, sys, json, shutil, re, textwrap, glob
from pathlib import Path

# ── Kaggle paths ──────────────────────────────────────────────────────────────
KAGGLE_INPUT = Path('/kaggle/input')
KAGGLE_WORK  = Path('/kaggle/working')
EXPORT_DIR   = KAGGLE_WORK / 'meme_model_export'
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

# ── Dataset paths ─────────────────────────────────────────────────────────────
MEMOTION_DIR = KAGGLE_INPUT / 'memotion-dataset-7k'
REDDIT_DIR   = KAGGLE_INPUT / 'reddit-memes-dataset'
TEMPLATE_DIR = KAGGLE_INPUT / 'meme-templates'

print('📂 Datasets found in /kaggle/input:')
for p in sorted(KAGGLE_INPUT.iterdir()):
    status = '✅' if p.is_dir() else '📄'
    print(f'   {status} {p.name}')

print(f'\n📁 Export dir: {EXPORT_DIR}')
print(f'📌 Memotion available: {MEMOTION_DIR.exists()}')
print(f'📌 Reddit memes available: {REDDIT_DIR.exists()}')
print(f'📌 Templates available:  {TEMPLATE_DIR.exists()}')

In [ ]:
# Kaggle already ships torch+cuda — only install extras
!pip install -q transformers==4.40.0 accelerate bitsandbytes sentencepiece
!pip install -q scikit-learn gradio pillow requests

import torch
print(f'✅ PyTorch  {torch.__version__}')
print(f'✅ CUDA     {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'   GPU:   {torch.cuda.get_device_name(0)}')
    print(f'   VRAM:  {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

In [ ]:
import pandas as pd
import numpy as np
import requests
from PIL import Image
from io import BytesIO
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

MEME_TEMPLATES = []   # {'name': str, 'path': str|None, 'url': str|None, 'source': str}

# ── Source 1: meme-templates dataset (local blank templates) ─────────────────
if TEMPLATE_DIR.exists():
    exts = ['*.jpg','*.jpeg','*.png','*.webp']
    tfiles = []
    for ext in exts:
        tfiles.extend(glob.glob(str(TEMPLATE_DIR/'**'/ext), recursive=True))
    for fp in tfiles:
        name = Path(fp).stem.replace('_',' ').replace('-',' ').title()
        MEME_TEMPLATES.append({'name': name, 'path': fp, 'url': None, 'source': 'kaggle-templates'})
    print(f'📦 kaggle-templates: {len(tfiles)} images loaded')

# ── Source 2: memotion-dataset-7k (memes with text labels) ───────────────────
if MEMOTION_DIR.exists():
    csvs = list(MEMOTION_DIR.glob('**/*.csv'))
    if csvs:
        df = pd.read_csv(csvs[0])
        print(f'📦 memotion: {len(df)} rows | cols: {list(df.columns)}')
        img_col   = next((c for c in df.columns if 'image' in c.lower() or 'file' in c.lower()), None)
        label_col = next((c for c in df.columns if 'text' in c.lower() or 'caption' in c.lower()), None)
        if img_col:
            n = 0
            for _, row in df.iterrows():
                for subdir in ['images','7k dataset','data','']:
                    p = MEMOTION_DIR / subdir / str(row[img_col]) if subdir else MEMOTION_DIR / str(row[img_col])
                    if p.exists():
                        lbl = str(row[label_col])[:60] if label_col else p.stem
                        MEME_TEMPLATES.append({'name': lbl, 'path': str(p), 'url': None, 'source': 'memotion'})
                        n += 1
                        break
            print(f'   Added {n} memotion images')

# ── Source 3: reddit-memes-dataset ───────────────────────────────────────────
if REDDIT_DIR.exists():
    rcsvs = list(REDDIT_DIR.glob('**/*.csv'))
    if rcsvs:
        rd = pd.read_csv(rcsvs[0])
        print(f'📦 reddit-memes: {len(rd)} rows | cols: {list(rd.columns)}')
        title_col = next((c for c in rd.columns if 'title' in c.lower() or 'name' in c.lower()), None)
        url_col   = next((c for c in rd.columns if 'url' in c.lower()), None)
        if title_col and url_col:
            for _, row in rd.head(200).iterrows():
                MEME_TEMPLATES.append({'name': str(row[title_col])[:60], 'path': None,
                                       'url': str(row[url_col]), 'source': 'reddit'})
            print(f'   Added {min(200, len(rd))} reddit meme URLs')

# ── Source 4: imgflip API fallback (always works) ─────────────────────────────
try:
    r = requests.get('https://api.imgflip.com/get_memes', timeout=10)
    for m in r.json()['data']['memes']:
        MEME_TEMPLATES.append({'name': m['name'], 'path': None, 'url': m['url'], 'source': 'imgflip'})
    print(f'🌐 imgflip API: {len(r.json()["data"]["memes"])} templates added')
except Exception as e:
    print(f'⚠️ imgflip API failed: {e}')

print(f'\n✅ Total templates: {len(MEME_TEMPLATES)}')

# Save URL-based templates for HuggingFace deployment
hf_templates = [t for t in MEME_TEMPLATES if t['url']]
with open(EXPORT_DIR / 'meme_templates.json', 'w') as f:
    json.dump(hf_templates, f, indent=2)
print(f'💾 Saved {len(hf_templates)} URL-templates → {EXPORT_DIR}/meme_templates.json')

In [ ]:
template_names = [t['name'] for t in MEME_TEMPLATES]
vectorizer    = TfidfVectorizer(ngram_range=(1,2), min_df=1)
tfidf_matrix  = vectorizer.fit_transform(template_names)


def find_best_template(context: str, top_k: int = 5) -> list:
    """Return top-k (template_dict, score) tuples for a text context."""
    qvec   = vectorizer.transform([context])
    scores = cosine_similarity(qvec, tfidf_matrix).flatten()
    idxs   = scores.argsort()[::-1][:top_k * 4]
    results = [(MEME_TEMPLATES[i], float(scores[i])) for i in idxs]

    # Prefer local Kaggle files first, then URL-based
    local = [(t,s) for t,s in results if t['path'] and Path(t['path']).exists()]
    urls  = [(t,s) for t,s in results if t['url']]
    combined = (local + urls)[:top_k]

    if not combined:
        # Full random fallback
        pop = [t for t in MEME_TEMPLATES if t['url']]
        combined = [(pop[i], 0.0) for i in np.random.choice(len(pop), min(top_k,len(pop)), replace=False)]
    return combined


def load_template_image(template: dict) -> Image.Image:
    """Load a meme template from local path or URL."""
    if template.get('path') and Path(template['path']).exists():
        return Image.open(template['path']).convert('RGB')
    elif template.get('url'):
        resp = requests.get(template['url'], timeout=15)
        return Image.open(BytesIO(resp.content)).convert('RGB')
    raise ValueError(f'No valid path or URL for: {template["name"]}')


# Test
matches = find_best_template('Monday mornings and coffee')
print('Test matches:')
for t, s in matches:
    print(f'  [{s:.3f}] "{t["name"]}"  ({t["source"]})')

In [ ]:
import torch
from transformers import Blip2Processor, Blip2ForConditionalGeneration

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
DTYPE  = torch.float16 if DEVICE == 'cuda' else torch.float32
print(f'Device: {DEVICE}  dtype: {DTYPE}')

#
# BLIP-2 options by GPU VRAM:
#   Salesforce/blip2-opt-2.7b       ->  ~6 GB   <-- default
#   Salesforce/blip2-flan-t5-xl     -> ~12 GB   (better reasoning)
#   Salesforce/blip2-flan-t5-xxl    -> ~24 GB   (needs A100)
#
BLIP2_ID        = 'Salesforce/blip2-opt-2.7b'
BLIP2_SAVE_PATH = EXPORT_DIR / 'blip2'

print(f'\n⏳ Loading {BLIP2_ID}...')
blip2_processor = Blip2Processor.from_pretrained(BLIP2_ID)
blip2_model     = Blip2ForConditionalGeneration.from_pretrained(
    BLIP2_ID, torch_dtype=DTYPE, device_map='auto'
)
blip2_model.eval()
print(f'✅ BLIP-2 loaded')

print(f'\n💾 Saving BLIP-2 → {BLIP2_SAVE_PATH}  ...')
blip2_processor.save_pretrained(str(BLIP2_SAVE_PATH))
blip2_model.save_pretrained(str(BLIP2_SAVE_PATH))
sz = sum(f.stat().st_size for f in BLIP2_SAVE_PATH.rglob('*') if f.is_file())
print(f'✅ BLIP-2 saved  ({sz/1e9:.2f} GB)')

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline, BitsAndBytesConfig

#
# Caption model options:
#   mistralai/Mistral-7B-Instruct-v0.2  -> best quality (default)
#   microsoft/phi-2                      -> lighter, ~6GB full
#   TinyLlama/TinyLlama-1.1B-Chat-v1.0  -> fastest, lower quality
#
CAPTION_ID        = 'mistralai/Mistral-7B-Instruct-v0.2'
CAPTION_SAVE_PATH = EXPORT_DIR / 'caption_model'

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16
)

print(f'⏳ Loading {CAPTION_ID} (4-bit quantized)...')
caption_tokenizer = AutoTokenizer.from_pretrained(CAPTION_ID)
caption_model     = AutoModelForCausalLM.from_pretrained(
    CAPTION_ID, quantization_config=bnb_config, device_map='auto'
)
caption_model.eval()

caption_pipe = pipeline(
    'text-generation',
    model=caption_model,
    tokenizer=caption_tokenizer,
    max_new_tokens=100,
    do_sample=True,
    temperature=0.85,
    top_p=0.9,
    return_full_text=False
)
print(f'✅ Mistral-7B loaded')

print(f'\n💾 Saving caption model → {CAPTION_SAVE_PATH} ...')
caption_tokenizer.save_pretrained(str(CAPTION_SAVE_PATH))
caption_model.save_pretrained(str(CAPTION_SAVE_PATH))
print(f'✅ Caption model saved')

In [ ]:
MEME_STYLES = {
    'funny':        'extremely funny with an unexpected punchline',
    'sarcastic':    'deeply sarcastic and dry',
    'motivational': 'over-the-top inspirational and motivational',
    'dark':         'dark humor, ironic, slightly edgy',
    'wholesome':    'wholesome, cute, warm-hearted',
    'relatable':    'super relatable everyday struggle'
}


def generate_meme_captions(context: str, style: str = 'funny') -> dict:
    """
    Use Mistral-7B to generate top-text + bottom-text for a meme.
    Returns: {'top': str, 'bottom': str}
    """
    style_desc = MEME_STYLES.get(style, MEME_STYLES['funny'])
    prompt = (
        f'[INST] You write meme captions. Style: {style_desc}.\n'
        f'Respond ONLY with valid JSON: {{"top": "...", "bottom": "..."}}\n'
        f'Max 8 words each. No markdown. No explanation.\n'
        f'Topic: {context} [/INST]'
    )
    raw = caption_pipe(prompt)[0]['generated_text'].strip()

    # Try JSON parse
    m = re.search(r'\{\s*"top"\s*:\s*"[^"]*"\s*,\s*"bottom"\s*:\s*"[^"]*"\s*\}', raw)
    if m:
        try:
            return json.loads(m.group())
        except:
            pass

    # Fallback: find any quoted strings
    quotes = [q for q in re.findall(r'"([^"]{3,60})"', raw)
              if q.lower() not in ['top', 'bottom']]
    return {
        'top':    quotes[0] if quotes else context[:40],
        'bottom': quotes[1] if len(quotes) > 1 else 'Story checks out'
    }


# Test all styles
test_topic = 'AI doing all the work while humans sleep'
print(f'Test topic: "{test_topic}"\n')
for s in MEME_STYLES:
    c = generate_meme_captions(test_topic, s)
    print(f'  [{s:<14}]  TOP: {c["top"]}  |  BOTTOM: {c["bottom"]}')

In [ ]:
def describe_image_blip2(image: Image.Image, question: str = None) -> str:
    """
    BLIP-2 caption OR visual QA.
    question=None  -> general caption
    question=str   -> VQA answer (PaLI / CoCa style)
    """
    if question:
        text   = f'Question: {question} Answer:'
        inputs = blip2_processor(image, text=text, return_tensors='pt').to(DEVICE, DTYPE)
    else:
        inputs = blip2_processor(image, return_tensors='pt').to(DEVICE, DTYPE)

    with torch.no_grad():
        ids = blip2_model.generate(**inputs, max_new_tokens=80, num_beams=4)
    return blip2_processor.batch_decode(ids, skip_special_tokens=True)[0].strip()


def load_image_from_source(source) -> Image.Image:
    """Load PIL Image from: file path | URL | PIL Image | bytes."""
    if isinstance(source, Image.Image):
        return source.convert('RGB')
    if isinstance(source, bytes):
        return Image.open(BytesIO(source)).convert('RGB')
    if isinstance(source, (str, Path)):
        s = str(source)
        if s.startswith('http'):
            return Image.open(BytesIO(requests.get(s, timeout=15).content)).convert('RGB')
        return Image.open(s).convert('RGB')
    raise ValueError(f'Unsupported source type: {type(source)}')


# Test on first Memotion image if available, else skip
test_img = None
if MEMOTION_DIR.exists():
    imgs = list(MEMOTION_DIR.glob('**/*.jpg'))[:1]
    if imgs:
        test_img = load_image_from_source(str(imgs[0]))
        print(f'Test image: {imgs[0]}')
        print('Caption:', describe_image_blip2(test_img))
        print('VQA:    ', describe_image_blip2(test_img, 'What is funny about this image?'))
if test_img is None:
    print('⚠️ No local image to test — BLIP-2 ready (add memotion-dataset-7k to see test)')

In [ ]:
from PIL import ImageDraw, ImageFont

_FONT_CANDIDATES = [
    '/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf',
    '/usr/share/fonts/truetype/liberation/LiberationSans-Bold.ttf',
    '/usr/share/fonts/truetype/freefont/FreeSansBold.ttf',
]

def get_font(size: int):
    for fp in _FONT_CANDIDATES:
        if os.path.exists(fp):
            return ImageFont.truetype(fp, size)
    return ImageFont.load_default()


def draw_meme_text(draw, text: str, position: str, w: int, h: int, fs: int = 42):
    """Classic white text + black outline at top or bottom of meme."""
    font  = get_font(fs)
    lines = textwrap.wrap(text.upper(), width=max(12, w // (fs // 2)))
    lh    = fs + 6
    y0    = 12 if position == 'top' else h - lh * len(lines) - 12

    for i, line in enumerate(lines):
        bbox = draw.textbbox((0,0), line, font=font)
        x = (w - (bbox[2] - bbox[0])) // 2
        y = y0 + i * lh
        # Outline
        for dx, dy in [(-2,0),(2,0),(0,-2),(0,2),(-2,-2),(2,-2),(-2,2),(2,2),(-3,0),(3,0)]:
            draw.text((x+dx, y+dy), line, font=font, fill='black')
        draw.text((x, y), line, font=font, fill='white')


def render_meme(template: dict, top_text: str, bottom_text: str,
                output_path: str = None, max_size: int = 720) -> Image.Image:
    """Overlay meme text on a template image. Returns PIL Image."""
    img = load_template_image(template)
    if max(img.size) > max_size:
        img.thumbnail((max_size, max_size), Image.LANCZOS)
    draw = ImageDraw.Draw(img)
    w, h = img.size
    fs   = max(24, w // 14)
    if top_text:    draw_meme_text(draw, top_text,    'top',    w, h, fs)
    if bottom_text: draw_meme_text(draw, bottom_text, 'bottom', w, h, fs)
    if output_path:
        img.save(output_path, quality=92)
    return img


print('✅ Meme renderer ready')

In [ ]:
import matplotlib.pyplot as plt

def generate_meme_from_text(
    user_text: str,
    style: str = 'funny',
    template_idx: int = 0,
    show: bool = True
) -> Image.Image:
    """
    TEXT → MEME full pipeline.
    
    Args:
        user_text:    Meme topic or phrase
        style:        funny | sarcastic | motivational | dark | wholesome | relatable
        template_idx: 0 = best match, 1-4 = alternatives
        show:         Display inline
    """
    print(f'\n📝 "{user_text}"  style={style}')

    templates    = find_best_template(user_text, top_k=5)
    template, sc = templates[min(template_idx, len(templates)-1)]
    print(f'   Template : "{template["name"]}"  score={sc:.3f}  [{template["source"]}]')

    captions = generate_meme_captions(user_text, style)
    print(f'   TOP      : {captions["top"]}')
    print(f'   BOTTOM   : {captions["bottom"]}')

    safe  = re.sub(r'[^\w]', '_', user_text[:30])
    fpath = str(KAGGLE_WORK / f'meme_{safe}_{style}.jpg')
    meme  = render_meme(template, captions['top'], captions['bottom'], fpath)

    if show:
        plt.figure(figsize=(7,7))
        plt.imshow(meme); plt.axis('off')
        plt.title(f'"{user_text}" | {style}', fontsize=10)
        plt.tight_layout(); plt.show()

    print(f'   ✅ Saved: {fpath}')
    return meme


# ════════════════════════════════════════════════
# 🎯 Change text and style below to try it out!
# ════════════════════════════════════════════════
generate_meme_from_text('When you fix one bug and create five more', 'sarcastic')
generate_meme_from_text('AI taking over the world',                  'dark')
generate_meme_from_text('Finally finishing the project on time',     'motivational')

In [ ]:
def generate_meme_from_image(
    image_source,
    style: str = 'funny',
    use_template: bool = True,
    vqa_question: str = None,
    show: bool = True
) -> Image.Image:
    """
    IMAGE → MEME full pipeline.

    Args:
        image_source:  Kaggle path | URL | PIL Image
        style:         Meme style
        use_template:  True = use a matched Kaggle template
                       False = caption the uploaded image directly
        vqa_question:  Custom question for BLIP-2 (optional)
        show:          Display inline
    """
    orig = load_image_from_source(image_source)
    print('\n📸 Image loaded')

    desc   = describe_image_blip2(orig)
    humor  = describe_image_blip2(orig, 'What is funny or ironic about this image?')
    scene  = describe_image_blip2(orig, vqa_question or 'Describe the main subject and mood.')
    ctx    = f'{desc}. {humor}. {scene}'

    print(f'   Caption : {desc}')
    print(f'   Humor   : {humor}')
    print(f'   Scene   : {scene}')

    captions = generate_meme_captions(ctx, style)
    print(f'   TOP     : {captions["top"]}')
    print(f'   BOTTOM  : {captions["bottom"]}')

    fpath = str(KAGGLE_WORK / f'meme_from_image_{style}.jpg')

    if use_template:
        template, _ = find_best_template(desc, top_k=1)[0]
        print(f'   Template: "{template["name"]}"  [{template["source"]}]')
        meme = render_meme(template, captions['top'], captions['bottom'], fpath)
    else:
        img = orig.copy()
        if max(img.size) > 720:
            img.thumbnail((720,720), Image.LANCZOS)
        draw = ImageDraw.Draw(img)
        w, h = img.size
        fs   = max(24, w // 14)
        draw_meme_text(draw, captions['top'],    'top',    w, h, fs)
        draw_meme_text(draw, captions['bottom'], 'bottom', w, h, fs)
        img.save(fpath, quality=92)
        meme = img

    if show:
        fig, axes = plt.subplots(1, 2, figsize=(14, 6))
        axes[0].imshow(orig);  axes[0].set_title('Input Image');       axes[0].axis('off')
        axes[1].imshow(meme);  axes[1].set_title(f'Meme ({style})');   axes[1].axis('off')
        plt.tight_layout(); plt.show()

    print(f'   ✅ Saved: {fpath}')
    return meme


# ════════════════════════════════════════════════════════════
# Example A: image from URL — caption overlaid on original
# ════════════════════════════════════════════════════════════
generate_meme_from_image(
    'https://upload.wikimedia.org/wikipedia/commons/thumb/4/43/Cute_dog.jpg/320px-Cute_dog.jpg',
    style='funny',
    use_template=False
)

In [ ]:
# ════════════════════════════════════════════════════════════
# Example B: random image from Memotion Kaggle dataset
# ════════════════════════════════════════════════════════════
if MEMOTION_DIR.exists():
    imgs = list(MEMOTION_DIR.glob('**/*.jpg'))[:50]
    if imgs:
        pick = str(np.random.choice([str(p) for p in imgs]))
        print(f'Using Kaggle image: {pick}')
        generate_meme_from_image(pick, style='relatable', use_template=True)
    else:
        print('No .jpg images found in memotion dataset directory.')
else:
    print('Add memotion-dataset-7k to Kaggle to test with dataset images.')

In [ ]:
def batch_meme_gallery(topics: list, styles: list = None, cols: int = 3):
    """Generate multiple memes and display as a grid."""
    if not styles:
        styles = list(MEME_STYLES.keys())

    generated = []
    for i, topic in enumerate(topics):
        style = styles[i % len(styles)]
        print(f'[{i+1}/{len(topics)}] "{topic}"  style={style}')
        try:
            tmpl, _  = find_best_template(topic, top_k=1)[0]
            caps     = generate_meme_captions(topic, style)
            out      = str(KAGGLE_WORK / f'batch_{i:03d}_{style}.jpg')
            meme_img = render_meme(tmpl, caps['top'], caps['bottom'], out)
            generated.append((meme_img, topic, style))
        except Exception as e:
            print(f'  ⚠️  {e}')

    rows = (len(generated) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(cols*5, rows*5))
    axes_flat = np.array(axes).flatten()

    for ax, (img, topic, style) in zip(axes_flat, generated):
        ax.imshow(img)
        ax.set_title(f'{topic[:25]}\n[{style}]', fontsize=8)
        ax.axis('off')
    for ax in axes_flat[len(generated):]:
        ax.axis('off')

    plt.suptitle('🎭 Meme Gallery', fontsize=14)
    plt.tight_layout()
    gpath = str(KAGGLE_WORK / 'meme_gallery.jpg')
    plt.savefig(gpath, dpi=120, bbox_inches='tight')
    plt.show()
    print(f'✅ Gallery: {gpath}')


batch_meme_gallery(
    topics=[
        'Debugging at 3am',
        'AI writing all my emails',
        'The meeting that was an email',
        'Stack Overflow is down',
        'Code works on first try',
        'Deploying on Friday afternoon'
    ],
    styles=['sarcastic','dark','funny','relatable','motivational','wholesome']
)

In [ ]:
# ── app.py for HuggingFace Spaces ────────────────────────────────────────────
app_py_content = r'''
import gradio as gr
import torch, json, re, textwrap, requests, os, random
from pathlib import Path
from PIL import Image, ImageDraw, ImageFont
from io import BytesIO
from transformers import (
    Blip2Processor, Blip2ForConditionalGeneration,
    AutoTokenizer, AutoModelForCausalLM, pipeline, BitsAndBytesConfig
)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE  = torch.float16 if DEVICE == "cuda" else torch.float32

print(f"Loading on {DEVICE}...")

blip2_processor = Blip2Processor.from_pretrained("blip2")
blip2_model = Blip2ForConditionalGeneration.from_pretrained(
    "blip2", torch_dtype=DTYPE, device_map="auto"
)
blip2_model.eval()

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                          bnb_4bit_compute_dtype=torch.float16)
cap_tok   = AutoTokenizer.from_pretrained("caption_model")
cap_model = AutoModelForCausalLM.from_pretrained(
    "caption_model", quantization_config=bnb, device_map="auto"
)
cap_pipe = pipeline("text-generation", model=cap_model, tokenizer=cap_tok,
                    max_new_tokens=100, do_sample=True, temperature=0.85,
                    return_full_text=False)

with open("meme_templates.json") as f:
    TEMPLATES = json.load(f)

STYLES = ["funny", "sarcastic", "motivational", "dark", "wholesome", "relatable"]
FONT_PATHS = ["/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf",
               "/usr/share/fonts/truetype/liberation/LiberationSans-Bold.ttf"]

def get_font(size):
    for fp in FONT_PATHS:
        if os.path.exists(fp): return ImageFont.truetype(fp, size)
    return ImageFont.load_default()

def draw_text(draw, text, pos, w, h, fs=40):
    font  = get_font(fs)
    text  = text.upper()
    lines = textwrap.wrap(text, width=max(12, w//(fs//2)))
    lh    = fs + 6
    y0    = 12 if pos == "top" else h - lh*len(lines) - 12
    for i, line in enumerate(lines):
        bbox = draw.textbbox((0,0), line, font=font)
        x = (w - (bbox[2]-bbox[0])) // 2
        y = y0 + i*lh
        for dx,dy in [(-2,0),(2,0),(0,-2),(0,2),(-2,-2),(2,-2),(-2,2),(2,2)]:
            draw.text((x+dx,y+dy), line, font=font, fill="black")
        draw.text((x,y), line, font=font, fill="white")

def blip2_caption(image, question=None):
    kw = {}
    if question:
        kw["text"] = f"Question: {question} Answer:"
    inputs = blip2_processor(image, return_tensors="pt", **kw).to(DEVICE, DTYPE)
    with torch.no_grad():
        ids = blip2_model.generate(**inputs, max_new_tokens=60)
    return blip2_processor.batch_decode(ids, skip_special_tokens=True)[0].strip()

def get_captions(ctx, style):
    prompt = (f'[INST] Write a {style} meme. '
              f'Return ONLY JSON: {{"top": "...", "bottom": "..."}} '
              f'Max 8 words each. Topic: {ctx} [/INST]')
    out = cap_pipe(prompt)[0]["generated_text"].strip()
    m = re.search(r'{\s*"top".*?}', out)
    if m:
        try: return json.loads(m.group())
        except: pass
    qs = [q for q in re.findall(r'"([^"]{3,60})"', out) if q.lower() not in ["top","bottom"]]
    return {"top": qs[0] if qs else ctx[:40], "bottom": qs[1] if len(qs)>1 else "..."}

def find_template(ctx):
    cl = ctx.lower()
    for t in TEMPLATES:
        if any(w in t["name"].lower() for w in cl.split() if len(w) > 3):
            return t
    return random.choice(TEMPLATES)

def generate_meme(text_in, img_in, style, use_own_img):
    if img_in is not None:
        pil = Image.fromarray(img_in).convert("RGB")
        ctx = blip2_caption(pil) + ". " + blip2_caption(pil, "What is funny here?")
    else:
        ctx = text_in or "something funny"
        pil = None

    caps = get_captions(ctx, style)

    if use_own_img and pil:
        base = pil.copy()
    else:
        tmpl = find_template(ctx)
        base = Image.open(BytesIO(requests.get(tmpl["url"], timeout=15).content)).convert("RGB")

    if max(base.size) > 720:
        base.thumbnail((720,720), Image.LANCZOS)
    draw = ImageDraw.Draw(base)
    w, h = base.size
    fs = max(24, w//14)
    draw_text(draw, caps["top"],    "top",    w, h, fs)
    draw_text(draw, caps["bottom"], "bottom", w, h, fs)
    return base, f"TOP: {caps['top']}\nBOTTOM: {caps['bottom']}"


with gr.Blocks(title="AI Meme Generator", theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 🎭 AI Meme Generator\nType a topic **or** upload an image to get a meme!")
    with gr.Row():
        with gr.Column(scale=1):
            txt   = gr.Textbox(label="Text topic", placeholder="e.g. Monday mornings and coffee")
            img   = gr.Image(label="Upload image (optional)")
            style = gr.Dropdown(STYLES, value="funny", label="Meme Style")
            own   = gr.Checkbox(label="Caption my image directly (skip template)", value=False)
            btn   = gr.Button("🎭 Generate!", variant="primary")
        with gr.Column(scale=1):
            out_img  = gr.Image(label="Your Meme")
            out_text = gr.Textbox(label="Generated Captions")
    btn.click(generate_meme, inputs=[txt, img, style, own], outputs=[out_img, out_text])
    gr.Examples(
        [["Code works but I don't know why", None, "sarcastic", False],
         ["AI replacing programmers",         None, "dark",      False],
         ["Finally fixed the bug",             None, "motivational", False]],
        inputs=[txt, img, style, own]
    )

demo.launch()
'''

with open(EXPORT_DIR / 'app.py', 'w') as f:
    f.write(app_py_content.strip())

# ── requirements.txt ─────────────────────────────────────────────────────────
with open(EXPORT_DIR / 'requirements.txt', 'w') as f:
    f.write('transformers>=4.40.0\naccelerator\nbitsandbytes\nsentencepiece\ntorch>=2.0.0\ntorchvision\npillow\nrequests\nscikit-learn\ngradio>=4.0.0\n')

# ── README.md / Space config ──────────────────────────────────────────────────
with open(EXPORT_DIR / 'README.md', 'w') as f:
    f.write(
        '---\n'
        'title: AI Meme Generator\n'
        'emoji: 🎭\n'
        'colorFrom: purple\n'
        'colorTo: pink\n'
        'sdk: gradio\n'
        'sdk_version: 4.0.0\n'
        'app_file: app.py\n'
        'pinned: false\n'
        '---\n\n'
        '# 🎭 AI Meme Generator\n\n'
        'Generate memes from text or images using BLIP-2 + Mistral-7B.\n\n'
        '## Kaggle Datasets Used\n'
        '- [Memotion 7K](https://www.kaggle.com/datasets/williamscott701/memotion-dataset-7k)\n'
        '- [Reddit Memes](https://www.kaggle.com/datasets/sayangoswami/reddit-memes-dataset)\n'
        '- [Meme Templates](https://www.kaggle.com/datasets/gmorinan/meme-templates)\n'
    )

print('✅ app.py written')
print('✅ requirements.txt written')
print('✅ README.md written')

In [ ]:
# ── Final Export Summary ──────────────────────────────────────────────────────
print('\n' + '='*65)
print('📦  EXPORT COMPLETE — Ready for HuggingFace Spaces')
print('='*65)

total = 0
for f in sorted(EXPORT_DIR.rglob('*')):
    if f.is_file():
        sz = f.stat().st_size
        total += sz
        print(f'  {str(f.relative_to(EXPORT_DIR)):<55} {sz/1e6:>8.1f} MB')

print(f'\n  Total size : {total/1e9:.2f} GB')
print(f'  Location   : {EXPORT_DIR}')

print('''
🚀  HOW TO DEPLOY ON HUGGINGFACE SPACES
──────────────────────────────────────────────────────────────
1. In Kaggle → Output tab → Download  meme_model_export/

2. Create a new Space:
   https://huggingface.co/new-space
   • SDK      : Gradio
   • Hardware : T4 GPU (free tier is fine)

3. Upload ALL files from meme_model_export/ to the Space root:
   ├── app.py
   ├── requirements.txt
   ├── README.md
   ├── meme_templates.json
   ├── blip2/               ← folder
   └── caption_model/       ← folder

4. Commit → Space auto-builds → share your URL!
──────────────────────────────────────────────────────────────
''')